In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# load files (they are in current working directory)
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')

# helper to convert epoch seconds to Asia/Shanghai tz-aware string
def ts_to_shanghai_str(ts):
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

# helper to create a compact preview of unique values (up to max_items)
def preview_uniques(values, max_items=50):
    uniques = pd.Series(values).dropna().astype(str).unique()
    total = len(uniques)
    if total == 0:
        return "", total
    preview_list = list(uniques[:max_items])
    preview_str = ", ".join(preview_list)
    if total > max_items:
        preview_str += f", ...(+{total-max_items} more)"
    return preview_str, total

# metric aggregates
metric_row_count = metric_df.shape[0]
metric_cmdb_preview, metric_cmdb_total = preview_uniques(metric_df['cmdb_id'].values, 50)
metric_kpi_preview, metric_kpi_total = preview_uniques(metric_df['kpi_name'].values, 50)
metric_min_ts = ts_to_shanghai_str(metric_df['timestamp'].min()) if metric_row_count>0 else None
metric_max_ts = ts_to_shanghai_str(metric_df['timestamp'].max()) if metric_row_count>0 else None

metric_summary = pd.DataFrame([{
    'file': 'metric.csv',
    'row_count': metric_row_count,
    'unique_cmdb_count': metric_cmdb_total,
    'unique_cmdb_preview': metric_cmdb_preview,
    'unique_kpi_count': metric_kpi_total,
    'unique_kpi_preview': metric_kpi_preview,
    'min_timestamp_shanghai': metric_min_ts,
    'max_timestamp_shanghai': metric_max_ts
}])

# trace aggregates
trace_row_count = trace_df.shape[0]
trace_cmdb_preview, trace_cmdb_total = preview_uniques(trace_df['cmdb_id'].values, 50)
trace_name_preview, trace_name_total = preview_uniques(trace_df['trace_name'].values, 50)
trace_min_ts = ts_to_shanghai_str(trace_df['timestamp'].min()) if trace_row_count>0 else None
trace_max_ts = ts_to_shanghai_str(trace_df['timestamp'].max()) if trace_row_count>0 else None

trace_summary = pd.DataFrame([{
    'file': 'trace.csv',
    'row_count': trace_row_count,
    'unique_cmdb_count': trace_cmdb_total,
    'unique_cmdb_preview': trace_cmdb_preview,
    'unique_trace_count': trace_name_total,
    'unique_trace_preview': trace_name_preview,
    'min_timestamp_shanghai': trace_min_ts,
    'max_timestamp_shanghai': trace_max_ts
}])

summary_df = pd.concat([metric_summary, trace_summary], ignore_index=True)
summary_df

```
Out[1]:
```
# build a concise plain-English summary using existing variables in the kernel
examples = lambda preview, n=5: ", ".join(preview.split(", ")[:n]) + ("..." if preview.count(",")>=n else "")

summary = (
    f"metric.csv: {metric_row_count} rows; {metric_cmdb_total} unique cmdb_id "
    f"(examples: {examples(metric_cmdb_preview)}); {metric_kpi_total} unique kpi_name "
    f"(examples: {examples(metric_kpi_preview)}); timestamp range (Asia/Shanghai): "
    f"{metric_min_ts} to {metric_max_ts}. "
    f"trace.csv: {trace_row_count} rows; {trace_cmdb_total} unique cmdb_id "
    f"(examples: {examples(trace_cmdb_preview)}); {trace_name_total} unique trace_name; "
    f"timestamp range (Asia/Shanghai): {trace_min_ts} to {trace_max_ts}."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

file  row_count  unique_cmdb_count                                unique_cmdb_preview  unique_kpi_count                                 unique_kpi_preview   min_timestamp_shanghai   max_timestamp_shanghai  unique_trace_count                               unique_trace_preview
0  metric.csv      90925                 56  db_001, db_002, db_003, db_004, db_005, db_006...             131.0  ACS, AIOS, AWS, CPU_Used_Pct, CPU_free_pct, Ca...  2020-05-23 04:00:00 CST  2020-05-23 05:00:00 CST                 NaN                                                NaN
1   trace.csv       6240                 10  docker_001, docker_002, docker_003, docker_004...               NaN                                                NaN  2020-05-23 04:30:00 CST  2020-05-23 04:59:00 CST                76.0  trace.from_os_022.remoteprocess.duration_mean,...```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# reuse metric_df if present; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

tz = pytz.timezone('Asia/Shanghai')

# ensure timestamp column is numeric and create tz-aware datetime column for Asia/Shanghai
metric_df['timestamp'] = metric_df['timestamp'].astype(int)
metric_df['ts_sh'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) compute global P95 per (cmdb_id, kpi_name) using entire metric_df (store in kernel memory)
p95_df = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

# 2) filter metric_df to incident window: 2020-05-23 04:30:00 to 2020-05-23 05:00:00 Asia/Shanghai (inclusive start, exclusive end typical)
start_win = pd.Timestamp('2020-05-23 04:30:00').tz_localize(tz)
end_win = pd.Timestamp('2020-05-23 05:00:00').tz_localize(tz)
window_df = metric_df[(metric_df['ts_sh'] >= start_win) & (metric_df['ts_sh'] <= end_win)].copy()

# 3) prepare per-(cmdb_id,kpi_name) window aggregates and anomaly counts
# merge global p95 into window points
window_with_p95 = window_df.merge(p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# mark anomalies where value >= global_p95
window_with_p95['is_anom'] = window_with_p95['value'] >= window_with_p95['global_p95']

# window aggregations: point count and max value
window_agg = (
    window_with_p95
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(window_point_count=('value', 'size'),
         max_value_in_window=('value', 'max'))
)

# anomaly aggregations: count and earliest anomaly timestamp
anom_agg = (
    window_with_p95[window_with_p95['is_anom']]
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(anomaly_count=('is_anom', 'sum'),
         earliest_anom_ts=('ts_sh', 'min'))
)

# assemble full table by starting from p95_df (covers all series)
full_tbl = p95_df.merge(window_agg, on=['cmdb_id', 'kpi_name'], how='left') \
                 .merge(anom_agg, on=['cmdb_id', 'kpi_name'], how='left')

# fill missing window counts with 0 and anomaly_count with 0
full_tbl['window_point_count'] = full_tbl['window_point_count'].fillna(0).astype(int)
full_tbl['anomaly_count'] = full_tbl['anomaly_count'].fillna(0).astype(int)

# format earliest anomaly timestamp in Asia/Shanghai string; NULL (None) if no anomaly
def fmt_ts(x):
    if pd.isna(x):
        return None
    return pd.to_datetime(x).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

full_tbl['earliest_anomaly_timestamp_in_window'] = full_tbl['earliest_anom_ts'].apply(fmt_ts)

# max_value_in_window: keep NULL if no points (window_point_count == 0), else the max value
full_tbl['max_value_in_window'] = full_tbl['max_value_in_window'].where(full_tbl['window_point_count']>0, other=None)

# Prepare final columns
final_cols = [
    'cmdb_id',
    'kpi_name',
    'global_p95',
    'window_point_count',
    'anomaly_count',
    'earliest_anomaly_timestamp_in_window',
    'max_value_in_window'
]
final_df = full_tbl[final_cols].copy()

# sorting: anomaly_count desc, then earliest_anomaly_timestamp_in_window asc (None last)
# create sorting key for earliest: convert strings back to datetime for correct ordering; None -> max
def to_sort_dt(s):
    if s is None:
        return pd.Timestamp.max.tz_localize(tz)
    return pd.to_datetime(s).tz_localize(None).tz_localize(tz) if isinstance(s, str) and s.endswith('CST')==False else pd.to_datetime(s)
# Simpler: convert existing earliest_anom_ts (datetime) directly from full_tbl if present
sort_key = full_tbl.get('earliest_anom_ts')
sort_key = sort_key.fillna(pd.Timestamp.max.tz_localize(tz))
final_df['_sort_earliest'] = sort_key

result_df = final_df.sort_values(by=['anomaly_count', '_sort_earliest'], ascending=[False, True]).drop(columns=['_sort_earliest']).reset_index(drop=True)

# limit to top 20 rows for display
top20_df = result_df.head(20)

# counts: total distinct series scanned and how many had anomaly_count>0
total_series = len(p95_df)
series_with_anomaly = int((final_df['anomaly_count'] > 0).sum())
counts_df = pd.DataFrame([{
    'total_series_scanned': total_series,
    'series_with_anomaly_count': series_with_anomaly
}])

# Display compact outputs: top20_df and counts_df
top20_df, counts_df

```
Out[2]:
```
```python
# Build a concise plain-English summary string using existing kernel results (top20_df, counts_df)
try:
    top20_df
    counts_df
except NameError:
    raise RuntimeError("Expected variables top20_df and counts_df in kernel.")

total_series = int(counts_df.at[0, 'total_series_scanned'])
series_with_anom = int(counts_df.at[0, 'series_with_anomaly_count'])

# prepare brief lines for top 6 affected series from top20_df
lines = []
for i, row in top20_df.head(6).iterrows():
    lines.append(
        f"{i+1}) {row['cmdb_id']} - {row['kpi_name']}: anomalies={int(row['anomaly_count'])}, "
        f"earliest={row['earliest_anomaly_timestamp_in_window']}, max_in_window={row['max_value_in_window']}"
    )

summary = (
    f"Scanned {total_series} distinct (cmdb_id, kpi_name) series. "
    f"{series_with_anom} series had >=1 point in the incident window exceeding their global P95.\n"
    "Top affected series (by anomaly_count):\n" + "\n".join(lines)
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id         kpi_name  global_p95  window_point_count  anomaly_count earliest_anomaly_timestamp_in_window  max_value_in_window
0    os_007      Disk_rd_ios    0.000000                  31             31              2020-05-23 04:30:00 CST             0.000000
1    os_020     Disk_io_util    0.000000                  31             31              2020-05-23 04:30:00 CST             0.333333
2   osb_001  app.succee_rate    1.000000                  31             31              2020-05-23 04:30:00 CST             1.000000
3    db_001             Hang    1.000000                  30             30              2020-05-23 04:30:00 CST             1.000000
4    db_001        MEM_Total  381.902264                  30             30              2020-05-23 04:30:00 CST           381.902264
5    db_001     On_Off_State    1.000000                  30             30              2020-05-23 04:30:00 CST             1.000000
6    db_001         Row_Lock    0.000000                  30             30              2020-05-23 04:30:00 CST             0.000000
7    db_001  SctRead_Per_Sec    0.000000                  30             30              2020-05-23 04:30:00 CST             0.000000
8    db_002             AIOS    0.000000                  30             30              2020-05-23 04:30:00 CST             0.010000
9    db_002             Hang    1.000000                  30             30              2020-05-23 04:30:00 CST             1.000000
10   db_002        MEM_Total  381.902264                  30             30              2020-05-23 04:30:00 CST           381.902264
11   db_002     On_Off_State    1.000000                  30             30              2020-05-23 04:30:00 CST             1.000000
12   db_002         Row_Lock    0.000000                  30             30              2020-05-23 04:30:00 CST             0.000000
13   db_002  SctRead_Per_Sec    0.000000                  30             30              2020-05-23 04:30:00 CST             0.000000
14   db_003             Hang    1.000000                  30             30              2020-05-23 04:30:00 CST             1.000000
15   db_003        MEM_Total  381.902264                  30             30              2020-05-23 04:30:00 CST           381.902264
16   db_003     On_Off_State    1.000000                  30             30              2020-05-23 04:30:00 CST             1.000000
17   db_003         Row_Lock    0.000000                  30             30              2020-05-23 04:30:00 CST             0.000000
18   db_004             AIOS    0.000000                  30             30              2020-05-23 04:30:00 CST             0.010000
19   db_004             Hang    1.000000                  30             30              2020-05-23 04:30:00 CST             1.000000,    total_series_scanned  series_with_anomaly_count
0                  2047                       1808)```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

# reuse trace_df if available
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

tz = pytz.timezone('Asia/Shanghai')

# ensure timestamp numeric and create Asia/Shanghai tz-aware timestamp column
trace_df['timestamp'] = trace_df['timestamp'].astype(int)
trace_df['ts_sh'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) compute global P95 per (cmdb_id, trace_name) using entire trace_df (store in kernel memory)
trace_p95_df = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

# 2) filter trace_df to the incident window: 2020-05-23 04:30:00 to 2020-05-23 05:00:00 Asia/Shanghai
start_win = pd.Timestamp('2020-05-23 04:30:00').tz_localize(tz)
end_win = pd.Timestamp('2020-05-23 05:00:00').tz_localize(tz)
trace_window_df = trace_df[(trace_df['ts_sh'] >= start_win) & (trace_df['ts_sh'] <= end_win)].copy()

# 3) merge global p95 into window points and compute anomalies
window_with_p95 = trace_window_df.merge(trace_p95_df, on=['cmdb_id', 'trace_name'], how='left')
window_with_p95['is_anom'] = window_with_p95['value'] >= window_with_p95['global_p95']

# window aggregates
window_agg = (
    window_with_p95
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(window_point_count=('value', 'size'),
         max_value_in_window=('value', 'max'))
)

# anomaly aggregates
anom_agg = (
    window_with_p95[window_with_p95['is_anom']]
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(anomaly_count=('is_anom', 'sum'),
         earliest_anom_ts=('ts_sh', 'min'))
)

# assemble full table starting from trace_p95_df (covers all series)
trace_full_tbl = trace_p95_df.merge(window_agg, on=['cmdb_id', 'trace_name'], how='left') \
                             .merge(anom_agg, on=['cmdb_id', 'trace_name'], how='left')

# fill missing window counts and anomaly counts
trace_full_tbl['window_point_count'] = trace_full_tbl['window_point_count'].fillna(0).astype(int)
trace_full_tbl['anomaly_count'] = trace_full_tbl['anomaly_count'].fillna(0).astype(int)

# format earliest anomaly timestamp string (Asia/Shanghai) or None
def fmt_ts(x):
    if pd.isna(x):
        return None
    return pd.to_datetime(x).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

trace_full_tbl['earliest_anomaly_timestamp_in_window'] = trace_full_tbl['earliest_anom_ts'].apply(fmt_ts)

# max_value_in_window: NULL if no points in window
trace_full_tbl['max_value_in_window'] = trace_full_tbl['max_value_in_window'].where(trace_full_tbl['window_point_count']>0, other=None)

# prepare final columns
final_cols = [
    'cmdb_id',
    'trace_name',
    'global_p95',
    'window_point_count',
    'anomaly_count',
    'earliest_anomaly_timestamp_in_window',
    'max_value_in_window'
]
final_df = trace_full_tbl[final_cols].copy()

# sorting: anomaly_count desc, earliest_anom_ts asc (None treated as far future)
max_future = pd.Timestamp('9999-12-31 23:59:59').tz_localize(tz)
sort_key = trace_full_tbl['earliest_anom_ts'].fillna(max_future)
final_df['_sort_earliest'] = sort_key

result_df = final_df.sort_values(by=['anomaly_count', '_sort_earliest'], ascending=[False, True]).drop(columns=['_sort_earliest']).reset_index(drop=True)

# top 20 rows for display
top20_df = result_df.head(20)

# counts
total_series = len(trace_p95_df)
series_with_anomaly = int((final_df['anomaly_count'] > 0).sum())
counts_df = pd.DataFrame([{
    'total_series_scanned': total_series,
    'series_with_anomaly_count': series_with_anomaly
}])

# Display compact outputs: top20_df and counts_df
top20_df, counts_df

```
Out[3]:
```
```python
# Build a concise plain-English summary string using existing kernel results (top20_df, counts_df)
try:
    top20_df
    counts_df
except NameError:
    raise RuntimeError("Expected variables top20_df and counts_df in kernel.")

total_series = int(counts_df.at[0, 'total_series_scanned'])
series_with_anom = int(counts_df.at[0, 'series_with_anomaly_count'])

# prepare brief lines for top 6 affected trace series from top20_df
lines = []
for rank, (_, row) in enumerate(top20_df.head(6).iterrows(), start=1):
    lines.append(
        f"{rank}) {row['cmdb_id']} - {row['trace_name']}: anomalies={int(row['anomaly_count'])}, "
        f"earliest={row['earliest_anomaly_timestamp_in_window']}, max_in_window={row['max_value_in_window']}"
    )

summary = (
    f"Scanned {total_series} distinct (cmdb_id, trace_name) series. "
    f"{series_with_anom} series had >=1 point in the incident window exceeding their global P95.\n"
    "Top affected trace series (by anomaly_count):\n" + "\n".join(lines)
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                    trace_name  global_p95  window_point_count  anomaly_count earliest_anomaly_timestamp_in_window  max_value_in_window
0   docker_001    trace.from_os_022.remoteprocess.error_rate         0.0                  30             30              2020-05-23 04:30:00 CST                  0.0
1   docker_001                     trace.self.csf.error_rate         0.0                  30             30              2020-05-23 04:30:00 CST                  0.0
2   docker_001               trace.self.flyremote.error_rate         0.0                  30             30              2020-05-23 04:30:00 CST                  0.0
3   docker_001                    trace.self.jdbc.error_rate         0.0                  30             30              2020-05-23 04:30:00 CST                  0.0
4   docker_001                   trace.self.local.error_rate         0.0                  30             30              2020-05-23 04:30:00 CST                  0.0
5   docker_001  trace.to_docker_007.remoteprocess.error_rate         0.0                  30             30              2020-05-23 04:30:00 CST                  0.0
6   docker_001  trace.to_docker_008.remoteprocess.error_rate         0.0                  30             30              2020-05-23 04:30:00 CST                  0.0
7   docker_002    trace.from_os_022.remoteprocess.error_rate         0.0                  30             30              2020-05-23 04:30:00 CST                  0.0
8   docker_002                     trace.self.csf.error_rate         0.0                  30             30              2020-05-23 04:30:00 CST                  0.0
9   docker_002               trace.self.flyremote.error_rate         0.0                  30             30              2020-05-23 04:30:00 CST                  0.0
10  docker_002                    trace.self.jdbc.error_rate         0.0                  30             30              2020-05-23 04:30:00 CST                  0.0
11  docker_002                   trace.self.local.error_rate         0.0                  30             30              2020-05-23 04:30:00 CST                  0.0
12  docker_002  trace.to_docker_007.remoteprocess.error_rate         0.0                  30             30              2020-05-23 04:30:00 CST                  0.0
13  docker_002  trace.to_docker_008.remoteprocess.error_rate         0.0                  30             30              2020-05-23 04:30:00 CST                  0.0
14  docker_003    trace.from_os_021.remoteprocess.error_rate         0.0                  30             30              2020-05-23 04:30:00 CST                  0.0
15  docker_003                     trace.self.csf.error_rate         0.0                  30             30              2020-05-23 04:30:00 CST                  0.0
16  docker_003               trace.self.flyremote.error_rate         0.0                  30             30              2020-05-23 04:30:00 CST                  0.0
17  docker_003                    trace.self.jdbc.error_rate         0.0                  30             30              2020-05-23 04:30:00 CST                  0.0
18  docker_003                   trace.self.local.error_rate         0.0                  30             30              2020-05-23 04:30:00 CST                  0.0
19  docker_003  trace.to_docker_005.remoteprocess.error_rate         0.0                  30             30              2020-05-23 04:30:00 CST                  0.0,    total_series_scanned  series_with_anomaly_count
0                   208                        208)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse trace_df if present, else load and prepare ts_sh
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')
if 'ts_sh' not in trace_df.columns:
    trace_df['timestamp'] = trace_df['timestamp'].astype(int)
    trace_df['ts_sh'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) global P95 per (cmdb_id, trace_name) using full file (store in kernel memory)
trace_p95_all = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

# substrings filter (case-insensitive)
subs = ['duration', 'duration_mean', 'latency', 'remoteprocess.duration']
mask_sub = trace_p95_all['trace_name'].str.lower().apply(lambda s: any(sub in s for sub in subs))
filtered_series = trace_p95_all[mask_sub].copy()  # series scanned after substring filter

# 2) filter trace_df to incident window
start_win = pd.Timestamp('2020-05-23 04:30:00').tz_localize(tz)
end_win = pd.Timestamp('2020-05-23 05:00:00').tz_localize(tz)
trace_window_df = trace_df[(trace_df['ts_sh'] >= start_win) & (trace_df['ts_sh'] <= end_win)].copy()

# further restrict window points to only the filtered trace_name series
# use lower-case matching for safety
name_mask = trace_window_df['trace_name'].str.lower().apply(lambda s: any(sub in s for sub in subs))
trace_window_df_filtered = trace_window_df[name_mask].copy()

# 3) merge global p95 into window points and compute aggregates
window_with_p95 = trace_window_df_filtered.merge(filtered_series[['cmdb_id','trace_name','global_p95']],
                                                 on=['cmdb_id','trace_name'], how='left')
window_with_p95['is_anom'] = window_with_p95['value'] >= window_with_p95['global_p95']

# window aggregates
window_agg = (
    window_with_p95
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(window_point_count=('value', 'size'),
         max_value_in_window=('value', 'max'))
)

# anomaly aggregates
anom_agg = (
    window_with_p95[window_with_p95['is_anom']]
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(anomaly_count=('is_anom', 'sum'),
         earliest_anom_ts=('ts_sh', 'min'))
)

# assemble full table starting from filtered_series to include series with zero points
trace_filtered_full = filtered_series.merge(window_agg, on=['cmdb_id','trace_name'], how='left') \
                                    .merge(anom_agg, on=['cmdb_id','trace_name'], how='left')

# fill defaults
trace_filtered_full['window_point_count'] = trace_filtered_full['window_point_count'].fillna(0).astype(int)
trace_filtered_full['anomaly_count'] = trace_filtered_full['anomaly_count'].fillna(0).astype(int)
trace_filtered_full['max_value_in_window'] = trace_filtered_full['max_value_in_window'].where(trace_filtered_full['window_point_count']>0, other=None)

# format earliest anomaly timestamp (Asia/Shanghai) or None
def fmt_ts(x):
    if pd.isna(x):
        return None
    return pd.to_datetime(x).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

trace_filtered_full['earliest_anomaly_timestamp_in_window'] = trace_filtered_full['earliest_anom_ts'].apply(fmt_ts)

# final columns
final_cols = [
    'cmdb_id',
    'trace_name',
    'global_p95',
    'window_point_count',
    'anomaly_count',
    'earliest_anomaly_timestamp_in_window',
    'max_value_in_window'
]
final_df = trace_filtered_full[final_cols].copy()

# 4) sort by max_value_in_window descending (None treated as -inf so they appear last)
sort_key = final_df['max_value_in_window'].fillna(-np.inf)
final_df['_sort_max'] = sort_key
result_df = final_df.sort_values(by=['_sort_max'], ascending=[False]).drop(columns=['_sort_max']).reset_index(drop=True)

top20_df = result_df.head(20)

# counts
total_series_filtered = len(filtered_series)
series_with_anom = int((final_df['anomaly_count'] > 0).sum())
counts_df = pd.DataFrame([{
    'total_series_scanned_filtered': total_series_filtered,
    'series_with_anomaly_count': series_with_anom
}])

top20_df, counts_df

```
Out[4]:
```
```python
# Build a concise plain-English summary string using existing kernel results (top20_df, counts_df)
try:
    top20_df
    counts_df
except NameError:
    raise RuntimeError("Expected variables top20_df and counts_df in kernel.")

total_series = int(counts_df.at[0, 'total_series_scanned_filtered'])
series_with_anom = int(counts_df.at[0, 'series_with_anomaly_count'])

lines = []
for rank, (_, row) in enumerate(top20_df.head(6).iterrows(), start=1):
    lines.append(
        f"{rank}) {row['cmdb_id']} - {row['trace_name']}: anomalies={int(row['anomaly_count'])}, "
        f"earliest={row['earliest_anomaly_timestamp_in_window']}, max={row['max_value_in_window']}"
    )

summary = (
    f"After filtering trace_name by the substrings, scanned {total_series} distinct (cmdb_id,trace_name) series. "
    f"{series_with_anom} of them had >=1 point in the incident window at-or-above their global P95.\n"
    "Top series by max_value_in_window (top 6 shown):\n" + "\n".join(lines)
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                        trace_name  global_p95  window_point_count  anomaly_count earliest_anomaly_timestamp_in_window  max_value_in_window
0       os_021                  trace.from_root.osb.duration_p95    9.795180                  30              2              2020-05-23 04:50:00 CST            14.664000
1       os_021                       trace.self.csf.duration_p95    9.777305                  30              2              2020-05-23 04:50:00 CST            14.644250
2       os_021    trace.to_docker_004.remoteprocess.duration_p95    9.984212                  30              2              2020-05-23 04:50:00 CST            14.094000
3   docker_004      trace.from_os_021.remoteprocess.duration_p95    9.984212                  30              2              2020-05-23 04:50:00 CST            14.094000
4   docker_003      trace.from_os_021.remoteprocess.duration_p95    9.573997                  30              2              2020-05-23 04:50:00 CST            13.766000
5       os_021    trace.to_docker_003.remoteprocess.duration_p95    9.573997                  30              2              2020-05-23 04:50:00 CST            13.766000
6   docker_004     trace.from_os_021.remoteprocess.duration_mean    5.290270                  30              2              2020-05-23 04:50:00 CST             7.093420
7       os_021   trace.to_docker_004.remoteprocess.duration_mean    5.290270                  30              2              2020-05-23 04:50:00 CST             7.093420
8       os_021                 trace.from_root.osb.duration_mean    5.231475                  30              2              2020-05-23 04:50:00 CST             6.584685
9       os_021                      trace.self.csf.duration_mean    5.203926                  30              2              2020-05-23 04:50:00 CST             6.559809
10      os_021   trace.to_docker_003.remoteprocess.duration_mean    4.875870                  30              2              2020-05-23 04:50:00 CST             6.241451
11  docker_003     trace.from_os_021.remoteprocess.duration_mean    4.875870                  30              2              2020-05-23 04:50:00 CST             6.241451
12  docker_004                       trace.self.csf.duration_p95    3.288975                  30              2              2020-05-23 04:50:00 CST             4.745900
13  docker_004    trace.to_docker_006.remoteprocess.duration_p95    2.587667                  30              2              2020-05-23 04:50:00 CST             4.445450
14  docker_006  trace.from_docker_004.remoteprocess.duration_p95    2.587667                  30              2              2020-05-23 04:50:00 CST             4.445450
15  docker_003                       trace.self.csf.duration_p95    2.783475                  30              2              2020-05-23 04:50:00 CST             4.314200
16  docker_003    trace.to_docker_006.remoteprocess.duration_p95    2.529860                  30              2              2020-05-23 04:50:00 CST             3.372300
17  docker_006  trace.from_docker_003.remoteprocess.duration_p95    2.529860                  30              2              2020-05-23 04:50:00 CST             3.372300
18      os_022    trace.to_docker_001.remoteprocess.duration_p95    2.607487                  30              2              2020-05-23 04:35:00 CST             2.699000
19  docker_001      trace.from_os_022.remoteprocess.duration_p95    2.607487                  30              2              2020-05-23 04:35:00 CST             2.699000,    total_series_scanned_filtered  series_with_anomaly_count
0                            104                        104)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse the previously stored per-(cmdb_id,kpi_name) metric table.
# Prefer 'full_tbl' (contains earliest_anom_ts datetime); fall back to 'final_df' if needed.
try:
    series_tbl = full_tbl.copy()
    # ensure expected columns exist
    if 'anomaly_count' not in series_tbl.columns:
        raise NameError
except NameError:
    # fallback
    series_tbl = final_df.copy()
    # try to recover earliest_anom_ts as datetime from string column if available
    if 'earliest_anomaly_timestamp_in_window' in series_tbl.columns:
        series_tbl['earliest_anom_ts'] = pd.to_datetime(
            series_tbl['earliest_anomaly_timestamp_in_window'].replace({None: pd.NaT})
        ).dt.tz_localize(tz, ambiguous='NaT', nonexistent='NaT')
    else:
        series_tbl['earliest_anom_ts'] = pd.NaT

# Filter to cmdb_id that start with 'db_'
db_series = series_tbl[series_tbl['cmdb_id'].astype(str).str.startswith('db_')].copy()

# Ensure numeric anomaly_count exists
db_series['anomaly_count'] = db_series['anomaly_count'].fillna(0).astype(int)

# Aggregations per db
def top_kpis_string(df, top_n=5):
    # df contains rows for one db; sort by anomaly_count desc then pick top kpis
    tmp = df.sort_values(by='anomaly_count', ascending=False)[['kpi_name','anomaly_count']].head(top_n)
    return ", ".join([f"{k} ({int(c)})" for k,c in zip(tmp['kpi_name'], tmp['anomaly_count'])])

agg_list = []
for db, g in db_series.groupby('cmdb_id'):
    num_kpi_series_scanned = g['kpi_name'].nunique()
    num_kpi_with_anomaly = int((g['anomaly_count'] > 0).sum())
    total_anomalous_points = int(g['anomaly_count'].sum())
    # earliest anomaly timestamp among kpis with anomaly_count>0
    if num_kpi_with_anomaly > 0:
        earliest_ts = g.loc[g['anomaly_count']>0, 'earliest_anom_ts'].min()
        if pd.isna(earliest_ts):
            earliest_str = None
        else:
            earliest_str = pd.to_datetime(earliest_ts).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    else:
        earliest_str = None
    top_kpis = top_kpis_string(g, top_n=5)
    agg_list.append({
        'cmdb_id': db,
        'num_kpi_series_scanned': int(num_kpi_series_scanned),
        'num_kpi_with_anomaly': int(num_kpi_with_anomaly),
        'total_anomalous_points': int(total_anomalous_points),
        'earliest_anomaly_timestamp': earliest_str,
        'top_kpis': top_kpis
    })

db_summary = pd.DataFrame(agg_list)

# sort by total_anomalous_points desc and limit to top 20
db_summary = db_summary.sort_values(by='total_anomalous_points', ascending=False).reset_index(drop=True)
top20_db_summary = db_summary.head(20)

# total number of distinct db_* cmdb_id scanned
total_db_count = int(db_summary['cmdb_id'].nunique())

# Display compact outputs: top20_db_summary and a small counts dataframe
counts_df = pd.DataFrame([{
    'total_db_cmdb_id_scanned': total_db_count
}])

top20_db_summary, counts_df

```
Out[5]:
```
```python
# Build a concise plain-English summary string using existing kernel results (top20_db_summary, counts_df)
try:
    top20_db_summary
    counts_df
except NameError:
    raise RuntimeError("Expected variables top20_db_summary and counts_df in kernel.")

total_db = int(counts_df.at[0, 'total_db_cmdb_id_scanned'])

lines = []
for i, row in top20_db_summary.head(6).iterrows():
    lines.append(
        f"{i+1}) {row['cmdb_id']}: scanned_kpis={row['num_kpi_series_scanned']}, "
        f"kpis_with_anomaly={row['num_kpi_with_anomaly']}, total_anomalous_points={row['total_anomalous_points']}, "
        f"earliest_anomaly={row['earliest_anomaly_timestamp']}, top_kpis=[{row['top_kpis']}]"
    )

summary = (
    f"Scanned {total_db} distinct db_* cmdb_id.\n"
    f"Top databases by total anomalous points (top 6 shown):\n" + "\n".join(lines)
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id  num_kpi_series_scanned  num_kpi_with_anomaly  total_anomalous_points earliest_anomaly_timestamp                                           top_kpis
0   db_011                      45                    40                     476    2020-05-23 04:30:00 CST  AIOS (30), Hang (30), Row_Lock (30), On_Off_St...
1   db_010                      45                    41                     411    2020-05-23 04:30:00 CST  ACS (30), AIOS (30), Hang (30), On_Off_State (...
2   db_004                      45                    42                     410    2020-05-23 04:30:00 CST  AIOS (30), Hang (30), Row_Lock (30), Proc_User...
3   db_012                      45                    41                     407    2020-05-23 04:30:00 CST  ACS (30), AIOS (30), Hang (30), On_Off_State (...
4   db_008                      45                    41                     387    2020-05-23 04:30:00 CST  Hang (30), SctRead_Per_Sec (30), Row_Lock (30)...
5   db_002                      45                    38                     381    2020-05-23 04:30:00 CST  AIOS (30), Hang (30), Row_Lock (30), On_Off_St...
6   db_006                      45                    38                     367    2020-05-23 04:30:00 CST  Hang (30), On_Off_State (30), Row_Lock (30), M...
7   db_013                      45                    38                     365    2020-05-23 04:30:00 CST  AIOS (30), Hang (30), Row_Lock (30), On_Off_St...
8   db_009                      45                    38                     346    2020-05-23 04:30:00 CST  AIOS (30), Hang (30), DFParaWrite_Per_Sec (30)...
9   db_001                      45                    43                     339    2020-05-23 04:30:00 CST  Hang (30), SctRead_Per_Sec (30), Row_Lock (30)...
10  db_007                      45                    42                     336    2020-05-23 04:30:00 CST  Hang (30), SctRead_Per_Sec (30), Row_Lock (30)...
11  db_005                      45                    43                     327    2020-05-23 04:30:00 CST  Hang (30), SctRead_Per_Sec (30), Row_Lock (30)...
12  db_003                      45                    41                     278    2020-05-23 04:30:00 CST  Hang (30), On_Off_State (30), Row_Lock (30), M...,    total_db_cmdb_id_scanned
0                        13)```
```